# Downloading dataset from Kaggle

In [11]:
import os
import shutil
import kagglehub
from dotenv import load_dotenv

In [ ]:
load_dotenv()

path = kagglehub.dataset_download("rtatman/questionanswer-dataset")

print("Path to dataset files:", path)

100%|██████████| 3.55M/3.55M [00:01<00:00, 3.45MB/s]

Extracting files...


Path to dataset files: C:\Users\Олег\.cache\kagglehub\datasets\rtatman\questionanswer-dataset\versions\1


In [ ]:
dataset_path = shutil.move(path, os.getcwd())
os.rename(dataset_path, "dataset")

'd:\\Programming\\Python\\Personal_projects\\RAG-backend\\1'

# Cleaning data

In [12]:
import pandas as pd
from pymongo import MongoClient

In [13]:
df = pd.read_csv("dataset\\S08_question_answer_pairs.txt", delimiter='\t')
df.head()

,ArticleTitle,Question,Answer,DifficultyFromQuestioner,DifficultyFromAnswerer,ArticleFile
0,Abraham_Lincoln,Was Abraham Lincoln the sixteenth President of...,yes,easy,easy,S08_set3_a4
1,Abraham_Lincoln,Was Abraham Lincoln the sixteenth President of...,Yes.,easy,easy,S08_set3_a4
2,Abraham_Lincoln,Did Lincoln sign the National Banking Act of 1...,yes,easy,medium,S08_set3_a4
3,Abraham_Lincoln,Did Lincoln sign the National Banking Act of 1...,Yes.,easy,easy,S08_set3_a4
4,Abraham_Lincoln,Did his mother die of pneumonia?,no,easy,medium,S08_set3_a4


In [18]:
df.isna().sum()

ArticleTitle                  0
Question                     19
Answer                      242
DifficultyFromQuestioner    491
DifficultyFromAnswerer      242
ArticleFile                   2
dtype: int64

In [20]:
df.dropna(axis=0, subset=['Question', 'Answer', 'ArticleFile'], inplace=True)
df.isna().sum()

ArticleTitle                  0
Question                      0
Answer                        0
DifficultyFromQuestioner    323
DifficultyFromAnswerer        1
ArticleFile                   0
dtype: int64

In [22]:
len(df['ArticleFile'].unique())

35

# Saving data as JSON

In [44]:
import json
import re
from pathlib import Path

In [38]:
datadir = Path(".\\dataset\\clean_data\\S08")
print(datadir)

dataset\clean_data\S08


In [62]:
topics = {}
def get_topic(article_name):
    set_index = int(re.match(r".*_set(\d+)_", article_name).group(1))
    article_index = int(re.match(r".*_a(\d+)$", article_name).group(1))
    if set_index not in topics:
        with open(f"dataset\\text_data\\S08_set{set_index}_topics.txt", "r") as f:
            topics[set_index] = f.readlines()
    return topics[set_index][article_index-1].strip().lower()

In [63]:
articles = []
for article_name in df['ArticleFile'].unique():
    article_path = "dataset\\text_data\\" + article_name + ".txt.clean"
    with open(article_path, "r", encoding='latin-1') as article_file:
        article = {'article': article_name, 'topic': get_topic(article_name).lower(), 'body': article_file.read()}
        articles.append(article)

datadir = Path(".\\dataset\\clean_data\\S08")
datadir.mkdir(parents=True, exist_ok=True)

with open("dataset\\clean_data\\S08\\articles.json", 'w') as f:
    json.dump(articles, f, indent=0)

In [ ]:
questions = []
for i, row in df.iterrows():
    question = {row['']}